In [ ]:
"""
 N=1 Scalar Multiplet with Boundary (Variance-Reduced)

Geometry:
- Point 1: z₁ = iε (near boundary)
- Point 2: z₂ = x + iε (same height, horizontal separation)
- Image:   z₁̄ = -iε (reflected below boundary)
- Separation: horizontal |x₁ - x₂|, with dy = 0
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
from scipy.optimize import curve_fit
from scipy.integrate import quad


def power_law(r, C, alpha):
    """Power law: C * r^alpha"""
    return C * r**alpha


def compute_fermion_deterministic(r_vals, k, log_vol):
    """
    Fermion ⟨ψ(z₁)ψ(z₂)⟩ near boundary with deterministic angular integration.
    
    Since dy = 0 (same height above boundary):
      phase_arg = kₓ·dx = k·cosθ·r
    
    Angular integral: ∫₀²π dθ/(2π) e^{-iθ} sin(kr cosθ) = J₁(kr)
    
    With amp=√k and amp²=k weighting:
      F(r) = mean_k[ k × J₁(kr) ] × log_vol
    """
    kr = k[:, None] * r_vals[None, :]
    j1 = jv(1, kr)
    fermion = np.mean(k[:, None] * j1, axis=0) * log_vol
    return fermion


def compute_boson_deterministic(r_vals, k, log_vol):
    """
    Boson ⟨∂φ(z₁)∂φ(z₂)⟩ near boundary with deterministic angular integration.
    
    Since dy = 0:
      phase_arg = kₓ·dx = k·cosθ·r
    
    Angular integral: ∫₀²π dθ/(2π) e^{-2iθ} cos(kr cosθ) = -J₂(kr)
    
    With amp=k (derivative) and amp²=k² weighting, and 4-clock giving 4× factor:
      B(r) = -4 × mean_k[ k² × J₂(kr) ] × log_vol
    """
    kr = k[:, None] * r_vals[None, :]
    j2 = jv(2, kr)
    boson = -4.0 * np.mean((k**2)[:, None] * j2, axis=0) * log_vol
    return np.real(boson)


def fermion_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff theory: F(r) = ∫_{k_min}^{k_max} J₁(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: jv(1, k * r), k_min, k_max, limit=1000)
        result[i] = val
    return result


def boson_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff theory: B(r) = -4 ∫_{k_min}^{k_max} k J₂(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: k * jv(2, k * r), k_min, k_max, limit=1000)
        result[i] = -4.0 * val
    return result


def super_boundary_multiplet_variance_reduced():
    """
    N=1 Scalar Multiplet with Boundary: Variance-Reduced Implementation
    
    Fermion: Deterministic Bessel J₁ (angular noise eliminated)
    Boson:   Deterministic Bessel J₂ (angular noise eliminated)
    Supercurrent: Wick factorization G = -F × B
    """
    
    print("=" * 70)
    print(" N=1 SCALAR MULTIPLET (VARIANCE-REDUCED)")
    print("=" * 70)
    print("\nMethod of Images with REAL separation (oscillatory regime)")
    print("Points near boundary: z₁ = iε, z₂ = x + iε")
    print("Fermion: Deterministic Bessel J₁")
    print("Boson: Deterministic Bessel J₂")
    print("Targets: Fermion 1/r, Boson 1/r², Supercurrent 1/r³\n")
    
    # Configuration (matched to bulk variance-reduced script)
    N_MODES = 30000
    N_REALIZATIONS = 800
    N_BLOCKS = 20
    k_min, k_max = 1e-3, 1e3
    
    # Geometry: Two points near boundary, vary horizontal separation
    epsilon = 0.05  # Small height above boundary
    x_separations = np.logspace(np.log10(0.05), np.log10(4.0), 50)
    
    # Separation: purely horizontal (dy = 0)
    dx = x_separations
    r_vals = x_separations  # |x₁ - x₂|
    
    # Normalization
    log_vol = np.log(k_max / k_min)
    
    print(f"Configuration:")
    print(f"  Modes: {N_MODES:,}")
    print(f"  k-range: [{k_min}, {k_max}] ({np.log10(k_max / k_min):.0f} decades)")
    print(f"  Realizations: {N_REALIZATIONS}")
    print(f"  Blocks: {N_BLOCKS} (for error estimation)")
    print(f"  Height above boundary: ε = {epsilon}")
    print(f"  Horizontal separation: x ∈ [{r_vals.min():.2f}, {r_vals.max():.2f}]")
    print(f"  Log-volume: {log_vol:.3f}\n")
    
    # Accumulators for all realizations
    fermion_all = np.zeros((N_REALIZATIONS, len(r_vals)))
    boson_all = np.zeros((N_REALIZATIONS, len(r_vals)))
    
    print("Simulating boundary correlations (deterministic Bessel)...")
    
    for real_idx in range(N_REALIZATIONS):
        if real_idx % 100 == 0:
            print(f"  Batch {real_idx}/{N_REALIZATIONS}")
        
        # Stratified k-sampling
        u = (np.arange(N_MODES) + np.random.uniform(size=N_MODES)) / N_MODES
        k = k_min * (k_max / k_min) ** u
        
        # Deterministic Bessel evaluation (angular noise eliminated!)
        fermion_all[real_idx] = compute_fermion_deterministic(r_vals, k, log_vol)
        boson_all[real_idx] = compute_boson_deterministic(r_vals, k, log_vol)
    
    print(f"  Done ({N_REALIZATIONS} realizations).")
    
    # ==================================================================
    # POST-PROCESSING WITH BLOCK-MEAN ERRORS
    # ==================================================================
    
    print("\nPost-processing with block-mean error estimation...")
    
    # Overall means
    F_mean = np.mean(fermion_all, axis=0)
    B_mean = np.mean(boson_all, axis=0)
    
    # Block-mean error estimation
    block_size = N_REALIZATIONS // N_BLOCKS
    F_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    B_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    G_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    
    for b in range(N_BLOCKS):
        i0 = b * block_size
        i1 = i0 + block_size
        F_blocks[b] = np.mean(fermion_all[i0:i1], axis=0)
        B_blocks[b] = np.mean(boson_all[i0:i1], axis=0)
        G_blocks[b] = -F_blocks[b] * B_blocks[b]
    
    F_sem = np.std(F_blocks, axis=0) / np.sqrt(N_BLOCKS)
    B_sem = np.std(B_blocks, axis=0) / np.sqrt(N_BLOCKS)
    
    # Wick's theorem combination
    G_mean = -F_mean * B_mean
    G_sem = np.std(G_blocks, axis=0) / np.sqrt(N_BLOCKS)
    
    # ==================================================================
    # FINITE-CUTOFF THEORY COMPARISON
    # ==================================================================
    
    print("Computing finite-cutoff theory predictions...")
    F_theory = fermion_theory_exact(r_vals, k_min, k_max)
    B_theory = boson_theory_exact(r_vals, k_min, k_max)
    G_theory = -F_theory * B_theory
    
    # Compute MC/theory ratios
    trust = (r_vals > 0.1) & (r_vals < 1.0)
    F_ratios = F_mean[trust] / F_theory[trust]
    B_ratios = B_mean[trust] / B_theory[trust]
    
    print(f"\n  Fermion MC/theory (r∈[0.1,1.0]): {np.mean(F_ratios):.4f} ± {np.std(F_ratios):.4f}")
    print(f"  Boson   MC/theory (r∈[0.1,1.0]): {np.mean(np.abs(B_ratios)):.4f} ± {np.std(B_ratios):.4f}")
    
    # ==================================================================
    # POWER LAW FITTING
    # ==================================================================
    
    print("\nFitting power laws...")
    
    results = {}
    
    for label, mask_range, trust_name in [
        ("OPE Window", (r_vals > 0.2) & (r_vals < 0.5), "OPE [0.2, 0.5]"),
        ("Standard", (r_vals > 0.1) & (r_vals < 2.0), "Standard [0.1, 2.0]"),
    ]:
        r_fit = r_vals[mask_range]
        
        print(f"\n  {trust_name}:")
        
        # Fermion
        try:
            popt_f, pcov_f = curve_fit(power_law, r_fit, np.abs(F_mean[mask_range]),
                                       p0=[0.2, -1.0], maxfev=5000)
            alpha_f = popt_f[1]
            alpha_f_err = np.sqrt(pcov_f[1, 1])
            acc_f = (1 - abs(alpha_f + 1)) * 100
            print(f"    Fermion:      α = {alpha_f:.4f} ± {alpha_f_err:.4f} (acc: {acc_f:.1f}%)")
        except Exception as e:
            alpha_f, acc_f = np.nan, 0
            print(f"    Fermion: FIT FAILED ({e})")
        
        # Boson
        try:
            popt_b, pcov_b = curve_fit(power_law, r_fit, np.abs(B_mean[mask_range]),
                                       p0=[1.0, -2.0], maxfev=5000)
            alpha_b = popt_b[1]
            alpha_b_err = np.sqrt(pcov_b[1, 1])
            acc_b = (1 - abs(alpha_b + 2) / 2) * 100
            print(f"    Boson:        α = {alpha_b:.4f} ± {alpha_b_err:.4f} (acc: {acc_b:.1f}%)")
        except Exception as e:
            alpha_b, acc_b = np.nan, 0
            print(f"    Boson: FIT FAILED ({e})")
        
        # Boson theory (finite-cutoff limit)
        try:
            popt_bt, _ = curve_fit(power_law, r_fit, np.abs(B_theory[mask_range]),
                                   p0=[1.0, -2.0], maxfev=5000)
            alpha_bt = popt_bt[1]
            acc_bt = (1 - abs(alpha_bt + 2) / 2) * 100
            print(f"    Theory limit:     {alpha_bt:.4f}         (acc: {acc_bt:.1f}%)")
        except Exception:
            alpha_bt = np.nan
        
        # Supercurrent
        try:
            popt_g, pcov_g = curve_fit(power_law, r_fit, np.abs(G_mean[mask_range]),
                                       p0=[1.0, -3.0], maxfev=5000)
            alpha_g = popt_g[1]
            alpha_g_err = np.sqrt(pcov_g[1, 1])
            acc_g = (1 - abs(alpha_g + 3) / 3) * 100
            print(f"    Supercurrent: α = {alpha_g:.4f} ± {alpha_g_err:.4f} (acc: {acc_g:.1f}%)")
        except Exception as e:
            alpha_g, acc_g = np.nan, 0
            print(f"    Supercurrent: FIT FAILED ({e})")
        
        if label == "OPE Window":
            results = {
                'fermion': {'alpha': alpha_f, 'accuracy': acc_f},
                'boson': {'alpha': alpha_b, 'accuracy': acc_b},
                'supercurrent': {'alpha': alpha_g, 'accuracy': acc_g},
            }
    
    # ==================================================================
    # SUMMARY
    # ==================================================================
    
    print("\n" + "=" * 70)
    print("N=1 SCALAR MULTIPLET WITH BOUNDARY (VARIANCE-REDUCED)")
    print("=" * 70)
    
    print(f"\nOPE Window (r ∈ [0.2, 0.5]):")
    print(f"  Fermion ⟨ψψ⟩:       α = {results['fermion']['alpha']:.3f}  "
          f"(target: -1.0, accuracy: {results['fermion']['accuracy']:.1f}%)")
    print(f"  Boson ⟨∂φ∂φ⟩:       α = {results['boson']['alpha']:.3f}  "
          f"(target: -2.0, accuracy: {results['boson']['accuracy']:.1f}%)")
    print(f"  Supercurrent ⟨GG⟩:  α = {results['supercurrent']['alpha']:.3f}  "
          f"(target: -3.0, accuracy: {results['supercurrent']['accuracy']:.1f}%)")
    
    print(f"\nFinite-Cutoff Theory Agreement:")
    print(f"  Fermion MC/theory: {np.mean(F_ratios):.4f} ± {np.std(F_ratios):.4f}")
    print(f"  Boson   MC/theory: {np.mean(np.abs(B_ratios)):.4f} ± {np.std(B_ratios):.4f}")
    
    # ==================================================================
    # PUBLICATION PLOTS
    # ==================================================================
    
    print("\nGenerating publication plots...")
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 5))
    
    # Theory curves (CFT prediction)
    theory_f_cft = 1 / (2 * np.pi * r_vals)
    theory_b_cft = 1 / (2 * np.pi * r_vals ** 2)
    theory_g_cft = 1 / (2 * np.pi * r_vals ** 3)
    
    # OPE window shading
    r_ope_lo, r_ope_hi = 0.2, 0.5
    
    # ---- Row 1: Correlator power laws ----
    
    # Panel 1: Fermion
    ax = axes[0]
    ax.loglog(r_vals, np.abs(F_mean), 'o-', label='Simulation', markersize=4, alpha=0.8)
    ax.loglog(r_vals, np.abs(F_theory), '--', label='Finite-cutoff theory',
              linewidth=2, alpha=0.7)
    ax.loglog(r_vals, theory_f_cft, ':', label=r'CFT: $1/(2\pi r)$',
              linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel('r', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|\langle\psi\psi\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(f'Boundary Fermion\n'
                 f'α = {results["fermion"]["alpha"]:.3f} '
                 f'(acc: {results["fermion"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Boson
    ax = axes[1]
    ax.loglog(r_vals, np.abs(B_mean), 's-', label='Simulation', markersize=4,
              alpha=0.8, color='C1')
    ax.loglog(r_vals, np.abs(B_theory), '--', label='Finite-cutoff theory',
              linewidth=2, alpha=0.7, color='C1')
    ax.loglog(r_vals, theory_b_cft, ':', label=r'CFT: $1/(2\pi r^2)$',
              linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel('r', fontsize=11, fontweight='bold')
    ax.set_ylabel(r'$|\langle\partial\phi\partial\phi\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(f'Boundary Boson (Bessel $J_2$)\n'
                 f'α = {results["boson"]["alpha"]:.3f} '
                 f'(acc: {results["boson"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 3: Supercurrent
    ax = axes[2]
    ax.loglog(r_vals, np.abs(G_mean), '^-', label='Simulation', markersize=4,
              alpha=0.8, color='C2')
    ax.loglog(r_vals, np.abs(G_theory), '--', label='Finite-cutoff theory',
              linewidth=2, alpha=0.7, color='C2')
    ax.loglog(r_vals, theory_g_cft, ':', label=r'CFT: $1/(2\pi r^3)$',
              linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel('r', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|\langle GG\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(f'Boundary Supercurrent (Wick)\n'
                 f'α = {results["supercurrent"]["alpha"]:.3f} '
                 f'(acc: {results["supercurrent"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('super_boundary_multiplet_variance_reduced.png', dpi=300, bbox_inches='tight')
    plt.savefig('super_boundary_multiplet_variance_reduced.pdf', bbox_inches='tight')
    
    print("\nPlots saved: super_boundary_multiplet_variance_reduced.png/pdf")
    print("\n✓ Boundary multiplet variance-reduced implementation complete!")
    
    return results


if __name__ == "__main__":
    results = super_boundary_multiplet_variance_reduced()


In [ ]:
"""
Boundary Conditions via Method of Images (Variance-Reduced)

Two independent experiments:
1. Bosonic Boundary (Neumann): Φ(z) = (1/√2)[φ(z) + φ(z̄)]
   - Auto-correlation ⟨|Φ(z)|²⟩ at z = iy along imaginary axis
   - Theory: G(z,z) ∝ -ln|z-z̄| = -ln(2y) (Neumann doubling)

2. Fermionic Boundary (Reflection): ψ_L = ψ_R at boundary
   - IMAGE GEOMETRY: ⟨ψ(z₁)ψ(z̄₂)⟩ using deterministic Bessel J₁
   - Points at z₁ = iε, z₂ = x + iε → image z̄₂ = x - iε
   - Image separation: √(x² + (2ε)²) → reduces to x for x >> ε
   - Theory: 1/(2πr) with r = horizontal separation
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.special import jv
from scipy.optimize import curve_fit
from scipy.integrate import quad


def power_law(r, C, alpha):
    """Power law: C * r^alpha"""
    return C * r**alpha


# ============================================================================
# EXPERIMENT 1: BOSONIC BOUNDARY (NEUMANN)
# ============================================================================

def compute_boundary_boson_vectorized(y_vals, n_modes, k_min, k_max):
    """
    One-point function ⟨|Φ(z)|²⟩ for Neumann boundary scalar.
    
    Φ(z) = (1/√2)[φ(z) + φ(z̄)] where z = iy (imaginary axis).
    
    For z = iy: φ(z) = Σ wᵢ exp(i(kₓ·0 + k_y·y)) = Σ wᵢ exp(i k_y y)
    Image φ(z̄) = Σ wᵢ exp(-i k_y y)
    So Φ(z) = (1/√2) Σ wᵢ 2cos(k_y y) = √2 Σ wᵢ cos(k_y y)
    
    ⟨|Φ|²⟩ = 2 Σ |wᵢ|² cos²(k_y,i y)  [for independent Gaussian weights]
    
    Instead of sampling Gaussian weights and computing |Φ|², we use the
    Wick contraction: ⟨|Φ|²⟩ = (2/N) Σᵢ cos²(k_{y,i} yⱼ) × log_vol
    
    Vectorized: no per-point inner loop.
    """
    # Stratified k-sampling
    u = (np.arange(n_modes) + np.random.uniform(size=n_modes)) / n_modes
    k_mag = k_min * (k_max / k_min) ** u
    
    # Random angles
    theta = np.random.uniform(0, 2 * np.pi, n_modes)
    
    # k_y component
    k_y = k_mag * np.sin(theta)
    
    # Vectorized: cos²(k_y·y) for all modes × all y-values
    # Shape: (n_modes,) × (n_y,) → (n_modes, n_y)
    cos2 = np.cos(k_y[:, None] * y_vals[None, :]) ** 2  # (n_modes, n_y)
    
    # Average over modes (Wick contraction) × 2 for boundary doubling
    log_vol = np.log(k_max / k_min)
    auto_corr = 2.0 * np.mean(cos2, axis=0) * log_vol
    
    return auto_corr


# ============================================================================
# EXPERIMENT 2: FERMIONIC BOUNDARY (REFLECTION) - Deterministic Bessel
# ============================================================================

def compute_boundary_fermion_deterministic(r_vals, k, log_vol):
    """
    Boundary fermionic propagator using deterministic Bessel J₁.
    
    The NMF propagator with antithetic pairing gives:
    ⟨ψ(0)ψ(r)⟩ = (1/2π) × mean_k[ k J₁(kr) ] × log_vol
    
    For the boundary theory, the reflection condition ψ_L = ψ_R at y=0
    identifies the left and right movers. The image propagator
    ⟨ψ(z₁)ψ(z̄₂)⟩ at the same height ε has the same 1/r structure
    since the image separation ≈ |x₁-x₂| for x >> ε.
    
    This uses the SAME deterministic Bessel approach as the bulk,
    since the horizontal separation makes k·r = k cosθ · r identical.
    """
    kr = k[:, None] * r_vals[None, :]
    j1 = jv(1, kr)
    # Factor 1/(2π) from angular normalization (matching Phase 1 NMF convention)
    fermion = np.mean(k[:, None] * j1, axis=0) * log_vol / (2 * np.pi)
    return fermion


def fermion_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff theory: F(r) = (1/2π) ∫_{k_min}^{k_max} J₁(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: jv(1, k * r), k_min, k_max, limit=1000)
        result[i] = val / (2 * np.pi)
    return result


# ============================================================================
# MAIN SIMULATION
# ============================================================================

def run_boundary_tests():
    
    print("=" * 80)
    print("BOUNDARY CONDITIONS VIA METHOD OF IMAGES (VARIANCE-REDUCED)")
    print("=" * 80)
    
    # ==================================================================
    # EXPERIMENT 1: BOSONIC BOUNDARY (NEUMANN)
    # ==================================================================
    
    print("\n" + "=" * 80)
    print("EXPERIMENT 1: BOSONIC BOUNDARY (NEUMANN)")
    print("=" * 80)
    
    # Configuration (improved from original)
    n_modes_boson = 20000      # 4× original
    n_real_boson = 400         # 4× original
    n_blocks_boson = 20
    k_min_b, k_max_b = 1e-2, 1e2  # 4 decades (appropriate for logarithmic scalar)
    
    y_vals = np.logspace(-1, 0.8, 30)  # Heights above boundary
    
    print(f"\nConfiguration:")
    print(f"  Modes: {n_modes_boson:,}")
    print(f"  k-range: [{k_min_b}, {k_max_b}]")
    print(f"  Realizations: {n_real_boson}")
    print(f"  Blocks: {n_blocks_boson}")
    print(f"  Heights: y ∈ [{y_vals.min():.2f}, {y_vals.max():.2f}]")
    
    # Accumulate all realizations
    boson_all = np.zeros((n_real_boson, len(y_vals)))
    
    print("\nRunning Bosonic Boundary simulation...")
    for i in range(n_real_boson):
        if i % 100 == 0:
            print(f"  Batch {i}/{n_real_boson}")
        boson_all[i] = compute_boundary_boson_vectorized(
            y_vals, n_modes_boson, k_min_b, k_max_b
        )
    
    # Block-mean error estimation
    block_size_b = n_real_boson // n_blocks_boson
    boson_blocks = np.zeros((n_blocks_boson, len(y_vals)))
    for b in range(n_blocks_boson):
        i0 = b * block_size_b
        i1 = i0 + block_size_b
        boson_blocks[b] = np.mean(boson_all[i0:i1], axis=0)
    
    boson_mean = np.mean(boson_all, axis=0)
    boson_sem = np.std(boson_blocks, axis=0) / np.sqrt(n_blocks_boson)
  
    log_vol_b = np.log(k_max_b / k_min_b)
    boson_theory = -1.0 * np.log(2 * y_vals)
    # Match at middle point for shape comparison
    mid_idx = len(y_vals) // 2
    boson_theory_shifted = boson_theory - boson_theory[mid_idx] + boson_mean[mid_idx]
    
    # Compute shape agreement: d(⟨|Φ|²⟩)/d(ln y) should be -1
    log_y = np.log(y_vals)
    dlog_y = np.diff(log_y)
    d_corr = np.diff(boson_mean)
    slope = d_corr / dlog_y
    mean_slope = np.mean(slope[3:-3])  # Exclude edges
    slope_target = -1.0
    slope_acc = (1 - abs(mean_slope - slope_target) / abs(slope_target)) * 100
    
    print(f"\n  Bosonic boundary results:")
    print(f"    Logarithmic slope: d⟨|Φ|²⟩/d(ln y) = {mean_slope:.3f} (target: -1.0)")
    print(f"    Slope accuracy: {slope_acc:.1f}%")
    
    # ==================================================================
    # EXPERIMENT 2: FERMIONIC BOUNDARY (REFLECTION)
    # ==================================================================
    
    print("\n" + "=" * 80)
    print("EXPERIMENT 2: FERMIONIC BOUNDARY (DETERMINISTIC BESSEL)")
    print("=" * 80)
    
    # Configuration (matched to bulk variance-reduced)
    n_modes_fermion = 30000
    n_real_fermion = 800
    n_blocks_fermion = 20
    k_min_f, k_max_f = 1e-3, 1e3
    
    r_vals = np.logspace(np.log10(0.05), np.log10(4.0), 40)
    log_vol_f = np.log(k_max_f / k_min_f)
    
    print(f"\nConfiguration:")
    print(f"  Modes: {n_modes_fermion:,}")
    print(f"  k-range: [{k_min_f}, {k_max_f}]")
    print(f"  Realizations: {n_real_fermion}")
    print(f"  Blocks: {n_blocks_fermion}")
    print(f"  Separations: r ∈ [{r_vals.min():.3f}, {r_vals.max():.2f}]")
    
    # Accumulate all realizations
    fermion_all = np.zeros((n_real_fermion, len(r_vals)))
    
    print("\nRunning Fermionic Boundary simulation (deterministic Bessel J₁)...")
    for i in range(n_real_fermion):
        if i % 100 == 0:
            print(f"  Batch {i}/{n_real_fermion}")
        
        # Stratified k-sampling
        u = (np.arange(n_modes_fermion) + np.random.uniform(size=n_modes_fermion)) / n_modes_fermion
        k = k_min_f * (k_max_f / k_min_f) ** u
        
        fermion_all[i] = compute_boundary_fermion_deterministic(r_vals, k, log_vol_f)
    
    print(f"  Done ({n_real_fermion} realizations).")
    
    # Block-mean errors
    block_size_f = n_real_fermion // n_blocks_fermion
    fermion_blocks = np.zeros((n_blocks_fermion, len(r_vals)))
    for b in range(n_blocks_fermion):
        i0 = b * block_size_f
        i1 = i0 + block_size_f
        fermion_blocks[b] = np.mean(fermion_all[i0:i1], axis=0)
    
    fermion_mean = np.mean(fermion_all, axis=0)
    fermion_sem = np.std(fermion_blocks, axis=0) / np.sqrt(n_blocks_fermion)
    
    # Finite-cutoff theory
    print("  Computing finite-cutoff theory...")
    fermion_theory = fermion_theory_exact(r_vals, k_min_f, k_max_f)
    
    # MC/theory ratio
    trust_f = (r_vals > 0.1) & (r_vals < 1.0)
    f_ratios = fermion_mean[trust_f] / fermion_theory[trust_f]
    
    print(f"\n  Fermion MC/theory (r∈[0.1,1.0]): {np.mean(f_ratios):.4f} ± {np.std(f_ratios):.4f}")
    
    # Power law fits
    print("\n  Power law fits:")
    results_fermion = {}
    for label, mask_range, trust_name in [
        ("OPE Window", (r_vals > 0.2) & (r_vals < 0.5), "OPE [0.2, 0.5]"),
        ("Standard", (r_vals > 0.1) & (r_vals < 2.0), "Standard [0.1, 2.0]"),
    ]:
        r_fit = r_vals[mask_range]
        try:
            popt, pcov = curve_fit(power_law, r_fit, np.abs(fermion_mean[mask_range]),
                                   p0=[0.2, -1.0], maxfev=5000)
            alpha = popt[1]
            alpha_err = np.sqrt(pcov[1, 1])
            acc = (1 - abs(alpha + 1)) * 100
            print(f"    {trust_name}: α = {alpha:.4f} ± {alpha_err:.4f} (acc: {acc:.1f}%)")
            if label == "OPE Window":
                results_fermion = {'alpha': alpha, 'accuracy': acc}
        except Exception as e:
            print(f"    {trust_name}: FIT FAILED ({e})")
            if label == "OPE Window":
                results_fermion = {'alpha': np.nan, 'accuracy': 0}
    
    # CFT theory line
    fermion_cft = 1 / (2 * np.pi * r_vals)
    
    # ==================================================================
    # VISUALIZATION
    # ==================================================================
    
    print("\n" + "=" * 80)
    print("CREATING PUBLICATION VISUALIZATION")
    print("=" * 80)
    
    fig = plt.figure(figsize=(16, 5))
    gs = GridSpec(1, 3, figure=fig, hspace=0.40, wspace=0.30)
    
    # ---- Panel 1: Bosonic boundary (auto-correlation) ----
    ax1 = fig.add_subplot(gs[0])
    
    ax1.errorbar(y_vals, boson_mean, yerr=boson_sem,
                 fmt='o-', linewidth=2, markersize=5, capsize=3,
                 color='#3498db', alpha=0.8, label='Neural $\\langle|\\Phi|^2\\rangle$')
    ax1.plot(y_vals, boson_theory_shifted, 'k--', linewidth=2.5,
             label='Theory: $-\\ln(2y)$ (shifted)', alpha=0.7)
    
    ax1.set_xlabel(r'$y_\perp$', fontsize=12)
    ax1.set_ylabel(r'Auto-correlation $\langle|\Phi(iy_\perp)|^2\rangle$', fontsize=12)
    ax1.set_title(f'Bosonic Neumann Boundary\n'
                  f'Slope: {mean_slope:.3f} \n(target: -1.0, acc: {slope_acc:.1f}%)',
                  fontsize=13, fontweight='bold')
    ax1.set_xscale('log')
    ax1.grid(True, alpha=0.3, which='both')
    ax1.legend(fontsize=10)
    
    # ---- Panel 2: Bosonic slope diagnostic ----
    ax2 = fig.add_subplot(gs[1])
    
    r_mid = np.sqrt(y_vals[:-1] * y_vals[1:])  # Geometric mean
    ax2.semilogx(r_mid, slope, 'o-', markersize=4, color='#3498db')
    ax2.axhline(-1.0, color='k', linestyle='--', linewidth=2, alpha=0.7,
                label='Target slope: -1.0')
    ax2.axhspan(-1.05, -0.95, alpha=0.1, color='green', label='5% band')
    
    ax2.set_xlabel(r'$y_\perp$', fontsize=12)
    ax2.set_ylabel(r'$d\langle|\Phi|^2\rangle / d(\ln y_\perp)$', fontsize=12)
    ax2.set_title('Bosonic Boundary: Logarithmic Slope', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, which='both')
    ax2.legend(fontsize=10)
    
    # ---- Panel 3: Fermionic boundary (power law) ----
    ax3 = fig.add_subplot(gs[2])
    
    r_ope_lo, r_ope_hi = 0.2, 0.5
    ax3.loglog(r_vals, np.abs(fermion_mean), 'o-', label='Simulation',
               markersize=4, alpha=0.8, color='#e74c3c')
    ax3.loglog(r_vals, np.abs(fermion_theory), '--', label='Finite-cutoff theory',
               linewidth=2, alpha=0.7, color='#e74c3c')
    ax3.loglog(r_vals, fermion_cft, ':', label='CFT: $1/(2\\pi r)$',
               linewidth=1.5, alpha=0.5, color='gray')
    ax3.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    
    ax3.set_xlabel('r', fontsize=12, fontweight='bold')
    ax3.set_ylabel('$|\\langle\\psi_L\\psi_R\\rangle|$', fontsize=12, fontweight='bold')
    alpha_f = results_fermion.get('alpha', np.nan)
    acc_f = results_fermion.get('accuracy', 0)
    ax3.set_title(f'Fermionic Boundary (Bessel $J_1$)\n'
                  f'α = {alpha_f:.3f} (acc: {acc_f:.1f}%)',
                  fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=10)

    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig('separate_boundary_tests_variance_reduced_1.png', dpi=300, bbox_inches='tight')
    plt.savefig('separate_boundary_tests_variance_reduced_1.pdf', bbox_inches='tight')
    
    # ==================================================================
    # SUMMARY
    # ==================================================================
    
    print("\n" + "=" * 80)
    print("BOUNDARY CONDITIONS SUMMARY (VARIANCE-REDUCED)")
    print("=" * 80)
    
    print(f"\n  Experiment 1 - Bosonic Neumann:")
    print(f"    Logarithmic slope: {mean_slope:.3f} (target: -1.0)")
    print(f"    Accuracy: {slope_acc:.1f}%")
    
    print(f"\n  Experiment 2 - Fermionic Reflection:")
    print(f"    OPE exponent: α = {alpha_f:.3f} (target: -1.0)")
    print(f"    Accuracy: {acc_f:.1f}%")
    print(f"    MC/theory: {np.mean(f_ratios):.4f} ± {np.std(f_ratios):.4f}")
    
    if slope_acc > 90 and acc_f > 90:
        print("\n  ✓✓ BOTH BOUNDARY EXPERIMENTS VALIDATED (>90%)")
    elif slope_acc > 85 and acc_f > 85:
        print("\n  ✓ BOUNDARY EXPERIMENTS WORKING (>85%)")
    else:
        print("\n  ~ PARTIAL VALIDATION - refinement needed")
    
    print("\n  Files saved:")
    print("    separate_boundary_tests_variance_reduced.png/pdf")
    print("=" * 80)
    
    return {
        'boson_slope': mean_slope,
        'boson_accuracy': slope_acc,
        'fermion_alpha': alpha_f,
        'fermion_accuracy': acc_f,
    }


if __name__ == "__main__":
    results = run_boundary_tests()


In [ ]:
"""
Super-Virasoro

Fermion: Holomorphic formulation (Phase 2 validated, 98% accuracy)
Boson:   Deterministic Bessel J₂ evaluation (analytic angular integration)
Combination: Wick's theorem factorization

Key Improvement: Analytic Integration of Angular Variable
- The 4-clock approach stochastically samples the angular direction
- The angular integral is known exactly: ∫ e^{-2iθ} cos(kr cosθ) dθ/(2π) = -J₂(kr)
- By evaluating J₂ directly, we eliminate the dominant noise source
- This is the "analytic Wick integration" strategy for the angular sector

Validation Strategy:
1. Power-law fit in OPE window (r ∈ [0.2, 0.5]) → exponent accuracy
2. MC/theory ratio across all r → confirms correct normalization
3. Block-mean error estimation → rigorous statistical uncertainties
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
from scipy.optimize import curve_fit
from scipy.integrate import quad

def power_law(r, C, alpha):
    """Power law: C * r^alpha"""
    return C * r**alpha

def compute_boson_deterministic(r_vals, k, log_vol):
    """
    Boson ⟨∂φ∂φ⟩ with deterministic angular integration.
    
    The spin-2 projected correlator with angular integral replaced by -J₂:
    B(r) = -4 × mean_k[ k² J₂(kr) ] × log_vol
    
    The factor 4 arises because the production code's 4-clock sums 4 terms
    without dividing by 4 (each quad contributes 4× the continuous integral).
    """
    kr = k[:, None] * r_vals[None, :]  # (N_modes, N_r)
    j2 = jv(2, kr)
    boson = -4.0 * np.mean((k**2)[:, None] * j2, axis=0) * log_vol
    return np.real(boson)

def compute_fermion_holomorphic(r_vals, k, theta, log_vol):
    """
    Fermion ⟨ψψ⟩ using holomorphic formulation (spin-1 projection).
    
    F(r) = mean_k[ k e^{-iθ} sin(kx r) ] × log_vol
    where kx = k cos(θ).
    """
    kx = k * np.cos(theta)
    sin_kr = np.sin(kx[:, None] * r_vals[None, :])
    weight = k * np.exp(-1j * theta)
    fermion = np.mean(weight[:, None] * sin_kr, axis=0) * log_vol
    return fermion

def compute_fermion_deterministic(r_vals, k, log_vol):
    """
    Fermion ⟨ψψ⟩ with deterministic angular integration.
    
    The angular integral: ∫₀²π dθ/(2π) e^{-iθ} sin(kr cosθ) = J₁(kr)
    
    Derivation: sin(z cosθ) = Im[e^{iz cosθ}], and
    ∫₀²π dθ/(2π) e^{iz cosθ} e^{-iθ} = i J₁(z)
    So ∫ dθ/(2π) e^{-iθ} sin(z cosθ) = Im[i J₁(z)] = J₁(z)  [J₁ real]
    
    The production code also has amp = √k with amp² weighting, giving
    F(r) = mean_k[ k × J₁(kr) ] × log_vol
    """
    kr = k[:, None] * r_vals[None, :]
    j1 = jv(1, kr)
    fermion = np.mean(k[:, None] * j1, axis=0) * log_vol
    return fermion

def boson_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff theory: B(r) = -4 ∫_{k_min}^{k_max} k J₂(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: k * jv(2, k*r), k_min, k_max, limit=1000)
        result[i] = -4.0 * val
    return result

def fermion_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff theory: F(r) = ∫_{k_min}^{k_max} J₁(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: jv(1, k*r), k_min, k_max, limit=1000)
        result[i] = val
    return result

def super_virasoro_variance_reduced():
    """
    Super-Virasoro: Variance-Reduced Implementation
    
    Fermion: Holomorphic NMF (98% validated)
    Boson: Deterministic Bessel J₂ (angular noise eliminated)
    Method: Wick Factorization
    """
    
    print("="*70)
    print("SUPER-VIRASORO ALGEBRA - VARIANCE REDUCED")
    print("="*70)
    print(f"\nFermion: Deterministic Bessel J₁ (analytic angular integration)")
    print("Boson: Deterministic Bessel J₂ (analytic angular integration)")
    print("Method: Wick Factorization G = -F × B\n")
    
    # Configuration
    N_MODES = 30000       # More modes for better k-integral convergence
    N_REALIZATIONS = 800  # Enough for sub-1% SEM
    N_BLOCKS = 20         # For block-mean error estimation
    k_min, k_max = 1e-3, 1e3
    
    # Geometry
    r_vals = np.logspace(np.log10(0.05), np.log10(4.0), 50)
    
    # Normalization
    log_vol = np.log(k_max / k_min)
    
    print(f"Configuration:")
    print(f"  Modes: {N_MODES:,}")
    print(f"  k-range: [{k_min}, {k_max}] ({np.log10(k_max/k_min):.0f} decades)")
    print(f"  Realizations: {N_REALIZATIONS}")
    print(f"  Blocks: {N_BLOCKS} (for error estimation)")
    print(f"  Log-volume: {log_vol:.3f}\n")
    
    # Accumulators for all realizations
    fermion_all = np.zeros((N_REALIZATIONS, len(r_vals)))
    boson_all = np.zeros((N_REALIZATIONS, len(r_vals)))
    
    print("Simulating realizations...")
    
    for real_idx in range(N_REALIZATIONS):
        if real_idx % 100 == 0:
            print(f"  Batch {real_idx}/{N_REALIZATIONS}")
        
        # Stratified k-sampling (shared for both sectors)
        u = (np.arange(N_MODES) + np.random.uniform(size=N_MODES)) / N_MODES
        k = k_min * (k_max / k_min)**u
        
        # FERMION: deterministic J₁ (angular noise eliminated!)
        fermion_all[real_idx] = compute_fermion_deterministic(r_vals, k, log_vol)
        
        # BOSON: deterministic J₂ (angular noise eliminated!)
        boson_all[real_idx] = compute_boson_deterministic(r_vals, k, log_vol)
    
    print(f"  Done ({N_REALIZATIONS} realizations).")
    
    # ==================================================================
    # POST-PROCESSING WITH BLOCK-MEAN ERRORS
    # ==================================================================
    
    print("\nPost-processing with block-mean error estimation...")
    
    # Overall means
    F_mean = np.mean(fermion_all, axis=0)
    B_mean = np.mean(boson_all, axis=0)
    
    # Block-mean error estimation
    block_size = N_REALIZATIONS // N_BLOCKS
    F_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    B_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    G_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    
    for b in range(N_BLOCKS):
        i0 = b * block_size
        i1 = i0 + block_size
        F_blocks[b] = np.mean(fermion_all[i0:i1], axis=0)
        B_blocks[b] = np.mean(boson_all[i0:i1], axis=0)
        G_blocks[b] = -F_blocks[b] * B_blocks[b]
    
    F_sem = np.std(F_blocks, axis=0) / np.sqrt(N_BLOCKS)
    B_sem = np.std(B_blocks, axis=0) / np.sqrt(N_BLOCKS)
    
    # Wick's theorem combination
    G_mean = -F_mean * B_mean
    G_sem = np.std(G_blocks, axis=0) / np.sqrt(N_BLOCKS)
    
    # ==================================================================
    # FINITE-CUTOFF THEORY COMPARISON
    # ==================================================================
    
    print("Computing finite-cutoff theory predictions...")
    B_theory = boson_theory_exact(r_vals, k_min, k_max)
    F_theory = fermion_theory_exact(r_vals, k_min, k_max)
    G_theory = -F_theory * B_theory
    
    # Compute MC/theory ratios in the trust region
    trust = (r_vals > 0.1) & (r_vals < 1.0)
    B_ratios = B_mean[trust] / B_theory[trust]
    F_ratios = F_mean[trust] / F_theory[trust]
    
    print(f"\n  Fermion MC/theory (r∈[0.1,1.0]): {np.mean(F_ratios):.4f} ± {np.std(F_ratios):.4f}")
    print(f"  Boson   MC/theory (r∈[0.1,1.0]): {np.mean(np.abs(B_ratios)):.4f} ± {np.std(B_ratios):.4f}")
    
    # ==================================================================
    # POWER LAW FITTING
    # ==================================================================
    
    print("\nFitting power laws...")
    
    results = {}
    
    for label, mask_range, trust_name in [
        ("OPE Window", (r_vals > 0.2) & (r_vals < 0.5), "OPE [0.2, 0.5]"),
        ("Standard",   (r_vals > 0.1) & (r_vals < 2.0), "Standard [0.1, 2.0]"),
    ]:
        r_fit = r_vals[mask_range]
        
        print(f"\n  {trust_name}:")
        
        # Fermion
        try:
            popt_f, pcov_f = curve_fit(power_law, r_fit, np.abs(F_mean[mask_range]), 
                                       p0=[0.2, -1.0], maxfev=5000)
            alpha_f = popt_f[1]
            alpha_f_err = np.sqrt(pcov_f[1, 1])
            acc_f = (1 - abs(alpha_f + 1)) * 100
            print(f"    Fermion:      α = {alpha_f:.4f} ± {alpha_f_err:.4f} (acc: {acc_f:.1f}%)")
        except Exception as e:
            alpha_f, acc_f = np.nan, 0
            print(f"    Fermion: FIT FAILED ({e})")
        
        # Boson
        try:
            popt_b, pcov_b = curve_fit(power_law, r_fit, np.abs(B_mean[mask_range]), 
                                       p0=[1.0, -2.0], maxfev=5000)
            alpha_b = popt_b[1]
            alpha_b_err = np.sqrt(pcov_b[1, 1])
            acc_b = (1 - abs(alpha_b + 2) / 2) * 100
            print(f"    Boson:        α = {alpha_b:.4f} ± {alpha_b_err:.4f} (acc: {acc_b:.1f}%)")
        except Exception as e:
            alpha_b, acc_b = np.nan, 0
            print(f"    Boson: FIT FAILED ({e})")
        
        # Boson theory (finite-cutoff limit)
        try:
            popt_bt, _ = curve_fit(power_law, r_fit, np.abs(B_theory[mask_range]), 
                                    p0=[1.0, -2.0], maxfev=5000)
            alpha_bt = popt_bt[1]
            acc_bt = (1 - abs(alpha_bt + 2) / 2) * 100
            print(f"    Theory limit:     {alpha_bt:.4f}         (acc: {acc_bt:.1f}%)")
        except:
            alpha_bt = np.nan
        
        # Supercurrent
        try:
            popt_g, pcov_g = curve_fit(power_law, r_fit, np.abs(G_mean[mask_range]), 
                                       p0=[1.0, -3.0], maxfev=5000)
            alpha_g = popt_g[1]
            alpha_g_err = np.sqrt(pcov_g[1, 1])
            acc_g = (1 - abs(alpha_g + 3) / 3) * 100
            print(f"    Supercurrent: α = {alpha_g:.4f} ± {alpha_g_err:.4f} (acc: {acc_g:.1f}%)")
        except Exception as e:
            alpha_g, acc_g = np.nan, 0
            print(f"    Supercurrent: FIT FAILED ({e})")
        
        if label == "OPE Window":
            results = {
                'fermion': {'alpha': alpha_f, 'accuracy': acc_f},
                'boson': {'alpha': alpha_b, 'accuracy': acc_b},
                'supercurrent': {'alpha': alpha_g, 'accuracy': acc_g},
            }
    
    # ==================================================================
    # SUMMARY
    # ==================================================================
    
    print("\n" + "="*70)
    print("SUPER-VIRASORO ALGEBRA VERIFICATION (Variance-Reduced)")
    print("="*70)
    
    print(f"\nOPE Window (r ∈ [0.2, 0.5]):")
    print(f"  Fermion ⟨ψψ⟩:       α = {results['fermion']['alpha']:.3f}  "
          f"(target: -1.0, accuracy: {results['fermion']['accuracy']:.1f}%)")
    print(f"  Boson ⟨∂φ∂φ⟩:       α = {results['boson']['alpha']:.3f}  "
          f"(target: -2.0, accuracy: {results['boson']['accuracy']:.1f}%)")
    print(f"  Supercurrent ⟨GG⟩:  α = {results['supercurrent']['alpha']:.3f}  "
          f"(target: -3.0, accuracy: {results['supercurrent']['accuracy']:.1f}%)")
    
    print(f"\nFinite-Cutoff Theory Agreement:")
    print(f"  Fermion MC/theory: {np.mean(F_ratios):.4f} ± {np.std(F_ratios):.4f}")
    print(f"  Boson   MC/theory: {np.mean(np.abs(B_ratios)):.4f} ± {np.std(B_ratios):.4f}")
    
    # ==================================================================
    # PUBLICATION PLOTS
    # ==================================================================
    
    print("\nGenerating publication plots...")
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 5))
    
    # ---- Row 1: Correlator power laws ----
    
    # Theory curves (CFT prediction, not finite-cutoff)
    theory_f_cft = 1 / (2 * np.pi * r_vals)
    theory_b_cft = 1 / (2 * np.pi * r_vals**2)
    theory_g_cft = 1 / (2 * np.pi * r_vals**3)
    
    # OPE window for shading
    r_ope_lo, r_ope_hi = 0.2, 0.5
    
    # Panel 1: Fermion
    ax = axes[0]
    ax.loglog(r_vals, np.abs(F_mean), 'o-', label='Simulation', markersize=4, alpha=0.8)
    ax.loglog(r_vals, np.abs(F_theory), '--', label='Finite-cutoff theory', 
              linewidth=2, alpha=0.7)
    ax.loglog(r_vals, theory_f_cft, ':', label=r'CFT: $1/(2\pi r)$', linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel(r'$r$', fontsize=12)
    ax.set_ylabel(r'$|\langle\psi\psi\rangle|$', fontsize=12)
    ax.set_title(f'Fermion Correlation\n'
                 f'α = {results["fermion"]["alpha"]:.3f} (acc: {results["fermion"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Boson
    ax = axes[1]
    ax.loglog(r_vals, np.abs(B_mean), 's-', label='Simulation', markersize=4, 
              alpha=0.8, color='C1')
    ax.loglog(r_vals, np.abs(B_theory), '--', label='Finite-cutoff theory', 
              linewidth=2, alpha=0.7, color='C1')
    ax.loglog(r_vals, theory_b_cft, ':', label=r'CFT: $1/(2\pi r^2)$', 
              linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel(r'$r$', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|\langle\partial\phi\partial\phi\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(f'Boson Derivative (Bessel $J_2$)\n'
                 f'α = {results["boson"]["alpha"]:.3f} (acc: {results["boson"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 3: Supercurrent
    ax = axes[2]
    ax.loglog(r_vals, np.abs(G_mean), '^-', label='Simulation', markersize=4, 
              alpha=0.8, color='C2')
    ax.loglog(r_vals, np.abs(G_theory), '--', label='Finite-cutoff theory', 
              linewidth=2, alpha=0.7, color='C2')
    ax.loglog(r_vals, theory_g_cft, ':', label=r'CFT: $1/(2\pi r^3)$', 
              linewidth=1.5, alpha=0.5)
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.1, color='green', label='OPE window')
    ax.set_xlabel(r'$r$', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|\langle GG\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(f'Supercurrent (Wick)\n'
                 f'α = {results["supercurrent"]["alpha"]:.3f} (acc: {results["supercurrent"]["accuracy"]:.1f}%)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    plt.savefig('super_virasoro_variance_reduced.png', dpi=300, bbox_inches='tight')
    plt.savefig('super_virasoro_variance_reduced.pdf', bbox_inches='tight')
    
    print("\nPlots saved: super_virasoro_variance_reduced.png/pdf")
    print("\n✓ Variance-reduced publication implementation complete!")
    
    return results

if __name__ == "__main__":
    results = super_virasoro_variance_reduced()


In [ ]:
"""
Stress Tensor OPE: Central Charge Extraction — Variance-Reduced

Computes ⟨T(z)T(0)⟩ = c/(2z⁴) for the free boson (c=1).

Method: Wick factorization via deterministic Bessel J₂

    ⟨TT⟩_CFT = B²/128

where B(r) = -4 · mean_k[k² J₂(kr)] · log_vol → -8/r² (infinite cutoff).

The finite-cutoff Bessel integral B(r) oscillates due to sharp UV truncation,
but the MC exactly reproduces the oscillatory theory prediction.
Central charge c=1 follows analytically from Gaussian regularization:

    B_smooth(r) = -4 ∫ k J₂(kr) e^{-(k/Λ)²} dk = -8/r²  (exact ∀Λ)

giving ⟨TT⟩ = B²/128 = 64/(128 r⁴) = 1/(2r⁴), i.e. c = 1.

Normalization chain:
    G_sim(r) = -ln(r)/(2L) + const
    ⟨JJ⟩_sim = 1/(4Lr²)                [→ B/(32L)]
    Z² = 4L                             [field renormalization]
    T_CFT = -½ Z² :J²_sim:
    ⟨TT⟩_CFT = (Z⁴/4)·2·⟨JJ⟩² = 1/(2r⁴)  →  c = 1 ✓
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.special import jv
from scipy.optimize import curve_fit
from scipy.integrate import quad
import os, sys, time



SCRIPT_DIR = os.getcwd()

DATA_FILE = os.path.join(SCRIPT_DIR, 'stress_tensor_mc_data.npz')
FIG_BASE = os.path.join(SCRIPT_DIR, 'stress_tensor_TT_variance_reduced')


# ============================================================================
# BESSEL COMPUTATION
# ============================================================================

def compute_B_deterministic(r_vals, k, log_vol):
    """
    B(r) = -4 · mean_k[k² J₂(kr)] · log_vol
    """
    kr = k[:, None] * r_vals[None, :]
    j2 = jv(2, kr)
    B = -4.0 * np.mean((k**2)[:, None] * j2, axis=0) * log_vol
    return np.real(B)


def B_theory_exact(r_vals, k_min, k_max):
    """Exact finite-cutoff B(r) = -4 ∫_{k_min}^{k_max} k J₂(kr) dk"""
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: k * jv(2, k * r), k_min, k_max, limit=5000)
        result[i] = -4.0 * val
    return result


def B_smooth_theory(r_vals, k_max):
    """
    Gaussian-regularized B(r) = -4 ∫ k J₂(kr) e^{-(k/Λ)²} dk = -8/r² exactly.
    """
    Lambda = k_max
    result = np.zeros(len(r_vals))
    for i, r in enumerate(r_vals):
        val, _ = quad(lambda k: k * jv(2, k * r) * np.exp(-(k / Lambda)**2),
                      0, 5 * Lambda, limit=5000)
        result[i] = -4.0 * val
    return result


def power_law(r, C, alpha):
    return C * r**alpha


# ============================================================================
# OUTPUT
# ============================================================================

_outfile = None

def pr(s=''):
    global _outfile
    print(s)
    sys.stdout.flush()
    if _outfile:
        _outfile.write(s + '\n')
        _outfile.flush()


# ============================================================================
# MAIN
# ============================================================================

def stress_tensor_TT():

    t_start = time.time()

    # Configuration
    k_min, k_max = 1e-3, 1e3
    log_vol = np.log(k_max / k_min)

    r_vals = np.logspace(np.log10(0.05), np.log10(4.0), 50)

    N_MODES = 30000
    N_REAL = 800
    N_BLOCKS = 20

    pr("=" * 70)
    pr("STRESS TENSOR OPE: CENTRAL CHARGE EXTRACTION")
    pr("=" * 70)
    pr(f"\nConfiguration:")
    pr(f"  k-range: [{k_min}, {k_max}] ({np.log10(k_max / k_min):.0f} decades)")
    pr(f"  log_vol L = {log_vol:.4f}")
    pr(f"  r-range: [{r_vals[0]:.3f}, {r_vals[-1]:.1f}] ({len(r_vals)} points)")
    pr(f"  Modes: {N_MODES:,}")
    pr(f"  Realizations: {N_REAL}")
    pr(f"  Blocks: {N_BLOCKS}")

    # ==================================================================
    # STEP 1: Compute B(r) — load cache or run MC
    # ==================================================================

    pr("\n" + "-" * 70)
    pr("STEP 1: B(r) via deterministic Bessel J_2")
    pr("-" * 70)

    if os.path.exists(DATA_FILE):
        pr(f"\n  Loading cached MC data from {DATA_FILE}...")
        data = np.load(DATA_FILE)
        B_all = data['B_all']
        r_check = data['r_vals']
        if B_all.shape == (N_REAL, len(r_vals)) and np.allclose(r_check, r_vals):
            pr(f"  Loaded {B_all.shape[0]} realizations x {B_all.shape[1]} r-pts")
        else:
            pr(f"  Cache mismatch, recomputing...")
            B_all = None
    else:
        B_all = None

    if B_all is None:
        B_all = np.zeros((N_REAL, len(r_vals)))
        pr(f"\n  Computing MC B(r)... ({N_REAL} x {N_MODES:,} modes)")
        t0 = time.time()

        for i in range(N_REAL):
            if i % 100 == 0:
                pr(f"    Realization {i}/{N_REAL}")
            u = (np.arange(N_MODES) + np.random.uniform(size=N_MODES)) / N_MODES
            k = k_min * (k_max / k_min) ** u
            B_all[i] = compute_B_deterministic(r_vals, k, log_vol)

        pr(f"  MC done in {time.time()-t0:.1f}s")

        # Cache for fast re-plotting
        np.savez(DATA_FILE, B_all=B_all, r_vals=r_vals)
        pr(f"  Saved MC data to {DATA_FILE}")

    B_mean = np.mean(B_all, axis=0)

    # Block-mean errors
    block_size = N_REAL // N_BLOCKS
    B_blocks = np.zeros((N_BLOCKS, len(r_vals)))
    for b in range(N_BLOCKS):
        i0 = b * block_size
        i1 = i0 + block_size
        B_blocks[b] = np.mean(B_all[i0:i1], axis=0)
    B_block_std = np.std(B_blocks, axis=0)

    # ==================================================================
    # STEP 2: Theory computations
    # ==================================================================

    pr("\n  Computing finite-cutoff theory B(r)...")
    t0 = time.time()
    B_theory = B_theory_exact(r_vals, k_min, k_max)
    pr(f"  Done in {time.time()-t0:.1f}s")

    pr("  Computing Gaussian-smoothed theory B(r)...")
    t0 = time.time()
    B_smooth = B_smooth_theory(r_vals, k_max)
    pr(f"  Done in {time.time()-t0:.1f}s")

    # Verify smoothed theory
    Br2_smooth = B_smooth * r_vals**2
    pr(f"\n  Gaussian-smoothed B(r)*r^2 (should be -8):")
    for idx in [0, 12, 24, 36, 49]:
        if idx < len(r_vals):
            pr(f"    r={r_vals[idx]:.3f}: {Br2_smooth[idx]:.6f}")

    # ==================================================================
    # STEP 3: B(r) MC/theory validation
    # ==================================================================

    pr("\n" + "-" * 70)
    pr("STEP 3: B(r) MC/theory validation")
    pr("-" * 70)

    # Compute ratio at multiple thresholds and in OPE windows
    for thresh_label, thresh_val in [("10", 10), ("100", 100)]:
        mask = np.abs(B_theory) > thresh_val
        n = np.sum(mask)
        if n > 0:
            ratio = B_mean[mask] / B_theory[mask]
            pr(f"\n  B MC/theory (|B_th| > {thresh_label}, n={n} pts):")
            pr(f"    mean  = {np.mean(ratio):.6f}")
            pr(f"    std   = {np.std(ratio):.6f}")
            pr(f"    range = [{np.min(ratio):.4f}, {np.max(ratio):.4f}]")
            if thresh_val == 100:
                b_ratio_100_mean = np.mean(ratio)
                b_ratio_100_std = np.std(ratio)

    # OPE window ratio (r < 0.3, where oscillation amplitude is small relative to signal)
    ope_ratio_mask = (r_vals < 0.3) & (np.abs(B_theory) > 10)
    n_ope = np.sum(ope_ratio_mask)
    if n_ope > 0:
        B_ratio_ope = B_mean[ope_ratio_mask] / B_theory[ope_ratio_mask]
        b_ratio_mean = np.mean(B_ratio_ope)
        b_ratio_std = np.std(B_ratio_ope)
        pr(f"\n  OPE window B MC/theory (r < 0.3, n={n_ope} pts):")
        pr(f"    mean  = {b_ratio_mean:.6f}")
        pr(f"    std   = {b_ratio_std:.6f}")
    else:
        b_ratio_mean, b_ratio_std = np.nan, np.nan

    # Power-law fits in multiple windows
    pr("\n  Power-law fits of |B(r)| → C·r^α:")
    fit_results = {}
    for win_label, r_lo, r_hi in [("OPE [0.05, 0.2]", 0.05, 0.2),
                                    ("Standard [0.1, 1.0]", 0.1, 1.0),
                                    ("Wide [0.05, 2.0]", 0.05, 2.0)]:
        mask = (r_vals >= r_lo) & (r_vals <= r_hi)
        r_w = r_vals[mask]
        try:
            popt, pcov = curve_fit(power_law, r_w, np.abs(B_mean[mask]),
                                   p0=[8.0, -2.0], maxfev=5000)
            alpha = popt[1]
            alpha_err = np.sqrt(pcov[1, 1])
            acc = (1 - abs(alpha + 2) / 2) * 100

            # Also fit theory in same window
            popt_t, _ = curve_fit(power_law, r_w, np.abs(B_theory[mask]),
                                  p0=[8.0, -2.0], maxfev=5000)
            alpha_th = popt_t[1]

            pr(f"    {win_label}: α_MC = {alpha:.4f} ± {alpha_err:.4f} "
               f"(acc {acc:.1f}%), α_theory = {alpha_th:.4f}")
            fit_results[win_label] = (popt, alpha, alpha_err, acc, alpha_th)
        except Exception as e:
            pr(f"    {win_label}: FIT FAILED ({e})")
            fit_results[win_label] = None

    # Use OPE window for primary slope
    ope_fit = fit_results.get("OPE [0.05, 0.2]")
    if ope_fit:
        popt_b, alpha_b, alpha_b_err, b_slope_acc, alpha_b_th = ope_fit
    else:
        popt_b = [np.nan, np.nan]
        alpha_b, alpha_b_err, b_slope_acc, alpha_b_th = np.nan, np.nan, 0, np.nan

    # ==================================================================
    # STEP 4: TT = B^2/128
    # ==================================================================

    pr("\n" + "-" * 70)
    pr("STEP 4: TT = B^2 / 128")
    pr("-" * 70)

    TT_mc = B_mean**2 / 128.0
    TT_theory = B_theory**2 / 128.0
    TT_smooth = B_smooth**2 / 128.0
    TT_cft = 0.5 / r_vals**4

    # TT ratio in OPE window
    tt_ope_mask = (r_vals < 0.3) & (np.abs(TT_theory) > 1.0)
    n_tt = np.sum(tt_ope_mask)
    if n_tt > 0:
        TT_ratio_ope = TT_mc[tt_ope_mask] / TT_theory[tt_ope_mask]
        tt_ratio_mean = np.mean(TT_ratio_ope)
        tt_ratio_std = np.std(TT_ratio_ope)
        pr(f"\n  TT MC/theory (r < 0.3, |TT_th| > 1, n={n_tt} pts):")
        pr(f"    mean  = {tt_ratio_mean:.6f}")
        pr(f"    std   = {tt_ratio_std:.6f}")
    else:
        tt_ratio_mean, tt_ratio_std = np.nan, np.nan

    # TT power-law fit in OPE window
    tt_fit_mask = (r_vals >= 0.05) & (r_vals <= 0.2)
    try:
        popt_tt, pcov_tt = curve_fit(power_law, r_vals[tt_fit_mask],
                                     TT_mc[tt_fit_mask],
                                     p0=[0.5, -4.0], maxfev=5000)
        alpha_tt = popt_tt[1]
        alpha_tt_err = np.sqrt(pcov_tt[1, 1])
        tt_slope_acc = (1 - abs(alpha_tt + 4) / 4) * 100

        popt_tt_th, _ = curve_fit(power_law, r_vals[tt_fit_mask],
                                  TT_theory[tt_fit_mask],
                                  p0=[0.5, -4.0], maxfev=5000)
        alpha_tt_th = popt_tt_th[1]

        pr(f"\n  TT power-law fit (r in [0.05, 0.2]):")
        pr(f"    α_MC     = {alpha_tt:.4f} ± {alpha_tt_err:.4f}  (target -4.0, acc {tt_slope_acc:.1f}%)")
        pr(f"    α_theory = {alpha_tt_th:.4f}")
    except Exception as e:
        alpha_tt, alpha_tt_err, tt_slope_acc = np.nan, np.nan, 0
        alpha_tt_th = np.nan
        popt_tt = [np.nan, np.nan]
        pr(f"  TT fit failed: {e}")

    # c from smooth theory
    c_smooth = TT_smooth / TT_cft
    pr(f"\n  c_eff from Gaussian-regularized theory:")
    pr(f"    (Should be 1.000000 at all r)")
    for idx in [0, 12, 24, 36, 49]:
        if idx < len(r_vals):
            pr(f"    r={r_vals[idx]:.3f}: c = {c_smooth[idx]:.6f}")

    # ==================================================================
    # SUMMARY
    # ==================================================================

    pr("\n" + "=" * 70)
    pr("STRESS TENSOR OPE: RESULTS SUMMARY")
    pr("=" * 70)

    pr(f"\n  Boson correlator B(r) [OPE window r ∈ [0.05, 0.2]]:")
    pr(f"    MC slope:        α = {alpha_b:.4f} ± {alpha_b_err:.4f}  "
       f"(target -2.0, acc {b_slope_acc:.1f}%)")
    pr(f"    Theory slope:    α = {alpha_b_th:.4f}")
    pr(f"    MC/theory ratio: {b_ratio_mean:.4f} ± {b_ratio_std:.4f}")

    pr(f"\n  Stress tensor ⟨TT⟩ = B²/128 [OPE window r ∈ [0.05, 0.2]]:")
    pr(f"    MC slope:        α = {alpha_tt:.4f} ± {alpha_tt_err:.4f}  "
       f"(target -4.0, acc {tt_slope_acc:.1f}%)")
    pr(f"    MC/theory ratio: {tt_ratio_mean:.4f} ± {tt_ratio_std:.4f}")

    pr(f"\n  Central charge (Gaussian regularization):")
    pr(f"    B_smooth(r) · r² = -8.000000  for all r  [Hankel transform identity]")
    pr(f"    ⟨TT⟩_smooth = B²/128 = 1/(2r⁴) exactly")
    pr(f"    ⟹  c = 1  (exact)")

    pr(f"\n  Total time: {time.time()-t_start:.1f}s")
    pr("=" * 70)

    # ==================================================================
    # PUBLICATION FIGURE (2×2)
    # ==================================================================

    pr("\nGenerating publication figure...")

    fig, axes = plt.subplots(1, 3, figsize=(12, 5))

    r_ope_lo, r_ope_hi = 0.05, 0.2

    # ---- Panel 1 (top-left): |B(r)| correlator ----
    ax = axes [0]

    ax.loglog(r_vals, np.abs(B_mean), 'o', label='MC (Bessel $J_2$)',
              markersize=4, alpha=0.8, color='C0', zorder=3)
    ax.loglog(r_vals, np.abs(B_theory), '-', label='Finite-cutoff theory',
              linewidth=1.5, alpha=0.6, color='C1')
    ax.loglog(r_vals, np.abs(B_smooth), '--', label=r'$8/r^2$ (Gaussian reg.)',
              linewidth=2, alpha=0.8, color='C2')
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.08, color='green', label='OPE window')

    ax.set_xlabel(r'$r$', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|B(r)|$', fontsize=12, fontweight='bold')
    ax.set_title(r'Boson Correlator $|B(r)|$',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

    # ---- Panel 2 (top-right): B MC/theory ratio ----
    ax = axes[1]

    # Only show points where theory is large enough for clean ratio
    for threshold, marker, ms, label, col in [
        (100, 'o', 5, r'$|B_{\rm th}| > 100$', 'C0'),
        (10, 's', 3, r'$|B_{\rm th}| > 10$', 'C3'),
    ]:
        mask = np.abs(B_theory) > threshold
        if np.sum(mask) > 0:
            ratio_pts = B_mean[mask] / B_theory[mask]
            ax.semilogx(r_vals[mask], ratio_pts, marker, markersize=ms,
                        alpha=0.7, label=label, color=col)

    ax.axhline(1.0, color='k', linestyle='--', linewidth=1, alpha=0.5)
    ax.axhspan(0.95, 1.05, alpha=0.1, color='green', label=r'5\% band')
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.08, color='green')

    ax.set_xlabel(r'$r$', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$B_{\rm MC} / B_{\rm theory}$', fontsize=12, fontweight='bold')
    ax.set_title(f'MC / Theory: ${b_ratio_mean:.4f} \\pm {b_ratio_std:.4f}$',
                 fontsize=12, fontweight='bold')
    ax.set_ylim(0.95, 1.05)
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(True, alpha=0.3)

    # ---- Panel 3 (bottom-left): |TT| = B^2/128 ----
    ax = axes[2]

    ax.loglog(r_vals, TT_mc, 'o', label=r'MC: $B_{\rm MC}^2/128$',
              markersize=4, alpha=0.7, color='C0', zorder=3)
    ax.loglog(r_vals, TT_theory, '-', label='Theory (finite cutoff)',
              linewidth=1.5, alpha=0.5, color='C1')
    ax.loglog(r_vals, TT_smooth, '--', label=r'$c/(2r^4)$, $c = 1$',
              linewidth=2.5, alpha=0.9, color='C2')
    ax.axvspan(r_ope_lo, r_ope_hi, alpha=0.08, color='green', label='OPE window')

    ax.set_xlabel(r'$r$', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$|\langle T(r)\, T(0)\rangle|$', fontsize=12, fontweight='bold')
    ax.set_title(r'$\langle TT \rangle = B^2/128$' + '\n'
                 + f'TT ratio: ${tt_ratio_mean:.4f} \\pm {tt_ratio_std:.4f}$',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIG_BASE + '.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIG_BASE + '.pdf', bbox_inches='tight')

    pr(f"\nPlots saved: {FIG_BASE}.png/pdf")
    pr("Done!")

    return {
        'alpha_b': alpha_b,
        'alpha_b_err': alpha_b_err,
        'b_slope_acc': b_slope_acc,
        'b_ratio_mean': b_ratio_mean,
        'b_ratio_std': b_ratio_std,
        'alpha_tt': alpha_tt,
        'alpha_tt_err': alpha_tt_err,
        'tt_slope_acc': tt_slope_acc,
    }

stress_tensor_TT()
